# Lección 3.2 — Función de pérdida: medir qué tan equivocado está el modelo

> **Módulo 3 · Cómo aprende un modelo** · Nivel 2 — Aprendiz
> ⏱️ Tiempo estimado: 80–100 min (lección ≈ 55 min · ejercicios finales ≈ 30 min)
> 📋 Requisitos previos: Lección 3.1 (el modelo como función con perillas).

### 🎯 Objetivos
Al terminar esta lección podrás:
- Explicar qué es una **función de pérdida** y por qué un modelo la necesita para aprender.
- Calcular a mano el **MAE** y el **MSE**, y predecir cuál reacciona más ante un dato atípico.
- Entender la intuición de la **entropía cruzada**: castigar a quien se equivoca *con mucha seguridad*.
- Ver la pérdida como un **paisaje** con un punto más bajo: el lugar adonde queremos llevar las perillas.

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar ni matemáticas. El código está para que **veas** los conceptos en acción; solo tendrás que cambiar números.

## ▶️ Preparación
Ejecuta esta celda una vez. Carga las herramientas y los datos de la heladería de la Lección 3.1.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (8, 4.5)

# Los mismos 30 días de la heladería (Lección 3.1)
rng = np.random.default_rng(0)
temperatura = rng.uniform(14, 36, 30).round(1)
ventas = (8 * temperatura - 60 + rng.normal(0, 15, 30)).round()

def modelo(temp, w, b):
    return w * temp + b

print("✅ Todo listo")

---
## 1. 📖 La intuición: un GPS para el error

Cuando usas un GPS, no te dice solo "vas mal". Te dice **"estás a 3.2 km"**. Ese número es lo que te permite saber si cada giro te acerca o te aleja.

Un modelo necesita lo mismo. Tras cada intento de perillas, necesita **un solo número** que diga *qué tan lejos* están sus predicciones de la realidad:

```
  perillas (w, b) ──► predicciones ──► comparar con la realidad ──► UN número: la PÉRDIDA
                                                                         │
                       "¿bajó o subió desde el intento anterior?" ◄──────┘
```

- **Pérdida alta** = el modelo se equivoca mucho.
- **Pérdida baja** = el modelo se parece a la realidad.
- **Entrenar** = buscar las perillas que hacen la pérdida **lo más baja posible**.

A esa regla para convertir "todos los errores" en un solo número se le llama **función de pérdida** (*loss function*) o **función de costo**.

### ✋ Ejercicio 1 — ¿Qué pronosticador es mejor?
Dos meteorólogos predijeron la temperatura de 5 días. Estos fueron sus errores (predicción − real), en °C:

| Día | Lun | Mar | Mié | Jue | Vie |
|---|---|---|---|---|---|
| Ana | +1 | −2 | +1 | −1 | +2 |
| Beto | 0 | 0 | +6 | 0 | −1 |

¿Quién es mejor? Inventa tú una forma de resumir cada fila en **un número** y decide.

✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

Depende de **cómo** resumas, y esa es justo la lección:
- **Promedio del tamaño del error** (sin signo): Ana = (1+2+1+1+2)/5 = **1.4 °C**; Beto = (0+0+6+0+1)/5 = **1.4 °C**. ¡Empate!
- Si **castigas más los errores grandes** (por ejemplo, elevando al cuadrado): Ana = (1+4+1+1+4)/5 = **2.2**; Beto = (0+0+36+0+1)/5 = **7.4**. Gana Ana.

Elegir la función de pérdida **es decidir qué errores te importan más**. Ahora le pondremos nombre a cada forma.
</details>

---
## 2. 📖 El concepto: del error de cada día a un solo número

Para cada ejemplo, el **error** (o **residuo**) es:

$$\text{error} = \text{predicción} - \text{valor real}$$

Hay que resumir todos los errores en un número. La idea más ingenua sería **promediarlos tal cual**…

### 🔮 Ejercicio 2 — La trampa del promedio
Un modelo se equivoca por **+100** helados el lunes y por **−100** el martes. ¿Cuál es el promedio de sus errores? ¿Es un buen modelo?

✍️ *Tu respuesta:*

In [ ]:
errores = np.array([+100, -100])
print("Promedio de los errores tal cual:", errores.mean())

<details><summary><b>👀 Ver respuesta</b></summary>

El promedio es **0**: ¡parece perfecto! Pero se equivocó por 100 helados *cada día*. Los errores positivos y negativos **se cancelan**. Por eso ninguna función de pérdida usa el promedio directo: hay que quitar el signo. Las dos formas más comunes son el **valor absoluto** (MAE) y el **cuadrado** (MSE).
</details>

### 📖 MAE y MSE

| Pérdida | Cómo se calcula | En palabras | Unidades |
|---|---|---|---|
| **MAE** (*Mean Absolute Error*, error absoluto medio) | promedio de \|error\| | "En promedio me equivoco por tantos helados" | helados |
| **MSE** (*Mean Squared Error*, error cuadrático medio) | promedio de error² | Igual, pero cada error se eleva al cuadrado: **los grandes pesan muchísimo más** | helados² 🤔 |

Como el MSE queda en "helados al cuadrado", a menudo se reporta su raíz, el **RMSE**, que vuelve a estar en helados (lo verás en la Lección 4.1).

Calculemos ambas para las perillas `w = 8`, `b = -60` de la lección anterior.

In [ ]:
def mae(real, pred):
    return np.mean(np.abs(pred - real))

def mse(real, pred):
    return np.mean((pred - real) ** 2)

pred = modelo(temperatura, w=8, b=-60)
print(f"MAE  = {mae(ventas, pred):.1f} helados")
print(f"MSE  = {mse(ventas, pred):.1f} helados²")
print(f"RMSE = {np.sqrt(mse(ventas, pred)):.1f} helados")

### 🔮 Ejercicio 3 — Doble error, ¿doble castigo?
Un error de **10** helados contra uno de **20**:
1. ¿Cuántas veces más castiga el MAE el error de 20?
2. ¿Y el MSE?

✍️ *Tu respuesta:*

In [ ]:
for e in [5, 10, 20, 40]:
    print(f"error {e:>2} → MAE lo cuenta como {e:>3}   ·   MSE lo cuenta como {e**2:>5}")

<details><summary><b>👀 Ver respuesta</b></summary>

1. El MAE castiga **2 veces** más (20 vs. 10): es proporcional.
2. El MSE castiga **4 veces** más (400 vs. 100). Duplicar el error **cuadruplica** el castigo.

Consecuencia: el MSE prefiere **muchos errores pequeños** antes que **unos pocos enormes**. Eso es bueno cuando los errores grandes son graves… y malo cuando son datos basura.
</details>

---
## 3. 📖 Qué pasa si hay un dato atípico

Un día de **34 °C** se descompuso el congelador y la heladería vendió **0 helados**. Ese día quedó en los datos.

Ahora buscaremos **las mejores perillas** de la recta de dos maneras: minimizando el MAE y minimizando el MSE. Para encontrarlas usamos fuerza bruta: probamos una cuadrícula de 40,000 combinaciones de `w` y `b` (con 2 perillas todavía se puede; en la Lección 3.3 veremos el método inteligente).

### 🔮 Ejercicio 4
¿Cuál de las dos rectas crees que se dejará "jalar" más por el día del congelador roto?

✍️ *Mi apuesta:* la recta del ___

In [ ]:
temp_atip = np.append(temperatura, 34)      # el día del congelador roto
ventas_atip = np.append(ventas, 0)

def mejores_perillas(perdida, x, y):
    # Fuerza bruta: prueba una cuadrícula de 200 × 200 combinaciones y devuelve la mejor
    mejor = (None, None, np.inf)
    for w in np.linspace(0, 15, 200):
        for b in np.linspace(-200, 100, 200):
            p = perdida(y, modelo(x, w, b))
            if p < mejor[2]:
                mejor = (w, b, p)
    return mejor

w_mae, b_mae, _ = mejores_perillas(mae, temp_atip, ventas_atip)
w_mse, b_mse, _ = mejores_perillas(mse, temp_atip, ventas_atip)

t = np.linspace(13, 37, 100)
plt.scatter(temp_atip, ventas_atip, edgecolor="k")
plt.scatter([34], [0], s=200, facecolor="none", edgecolor="red", lw=2, label="Congelador roto")
plt.plot(t, modelo(t, w_mae, b_mae), lw=2, label=f"Mejor según MAE: w={w_mae:.1f}, b={b_mae:.0f}")
plt.plot(t, modelo(t, w_mse, b_mse), lw=2, label=f"Mejor según MSE: w={w_mse:.1f}, b={b_mse:.0f}")
plt.xlabel("Temperatura (°C)"); plt.ylabel("Helados"); plt.legend(); plt.grid(alpha=.3); plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

La recta del **MSE** se inclina menos: el punto en 0 helados la jala hacia abajo, porque un error enorme **al cuadrado** pesa más que todos los demás juntos. La del **MAE** casi no se inmuta: para ella ese punto es "un error más".

Regla práctica:
- ¿Los atípicos son **errores de medición** o casos que no te importan? → **MAE** (o *Huber*, un punto medio).
- ¿Los errores grandes son **especialmente graves** (p. ej. quedarte sin inventario en el día más caluroso)? → **MSE**.
</details>

---
## 4. 📖 El paisaje de la pérdida

Aquí está la idea más importante de la lección. Para **cada** posición de las perillas, la pérdida da un número. Si dibujamos ese número para muchas posiciones, obtenemos un **paisaje**.

Empecemos con una sola perilla: dejamos `b` fijo en −56 y movemos `w` de 0 a 16.

In [ ]:
ws = np.linspace(0, 16, 200)
perdidas = [mse(ventas, modelo(temperatura, w, -56)) for w in ws]

plt.plot(ws, perdidas, lw=2)
plt.xlabel("Perilla w"); plt.ylabel("Pérdida (MSE)")
plt.title("Paisaje de la pérdida al mover solo w"); plt.grid(alpha=.3); plt.show()

### ✋ Ejercicio 5 — Lee el paisaje
1. ¿Qué forma tiene la curva?
2. ¿Cerca de qué valor de `w` está el punto más bajo? ¿Coincide con lo que encontraste a mano en la Lección 3.1?
3. Si estás en `w = 12`, ¿hacia dónde tendrías que mover la perilla para bajar la pérdida? ¿Y si estás en `w = 3`?

✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. Un **valle** o tazón: baja, toca fondo y vuelve a subir.
2. El fondo está en **`w ≈ 8`**, el mismo valor que encontraste moviendo perillas a mano.
3. En `w = 12` el terreno **sube hacia la derecha**, así que hay que mover `w` hacia la **izquierda** (bajarla). En `w = 3` el terreno sube hacia la izquierda, así que hay que **subirla**.

👉 Esa pregunta, *"¿hacia dónde baja el terreno desde donde estoy?"*, es todo el descenso de gradiente (Lección 3.3).
</details>

### 📖 Con dos perillas: un mapa topográfico
Si movemos `w` **y** `b` a la vez, el paisaje es una superficie con montañas y valles. Lo dibujamos como un **mapa topográfico**: cada línea une puntos con la misma pérdida, como las curvas de nivel de un mapa de montaña. El color oscuro es el fondo del valle.

In [ ]:
W, B = np.meshgrid(np.linspace(2, 14, 150), np.linspace(-200, 90, 150))
Z = np.array([[mse(ventas, modelo(temperatura, w, b)) for w, b in zip(fw, fb)] for fw, fb in zip(W, B)])

plt.contourf(W, B, np.log10(Z), levels=30, cmap="viridis")
plt.colorbar(label="log₁₀(pérdida): más oscuro = más bajo")
plt.contour(W, B, np.log10(Z), levels=15, colors="white", linewidths=.4)
plt.plot(8, -60, "r*", ms=15, label="Tus perillas de la 3.1 (8, −60)")
plt.xlabel("Perilla w"); plt.ylabel("Perilla b"); plt.legend(); plt.title("Mapa del paisaje de la pérdida")
plt.show()

### ✋ Ejercicio 6
1. ¿Qué forma tiene el fondo del valle: un pozo redondo o un **cañón** largo y estrecho?
2. Fíjate en la orientación del cañón: si subes `w`, ¿qué tienes que hacer con `b` para seguir en el fondo? Piensa en la recta: si la inclinas más, ¿qué le pasa a la predicción?

✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. Un **cañón diagonal**, largo y estrecho: hay muchas combinaciones de `w` y `b` con pérdida *casi* igual de baja.
2. Si subes `w` tienes que **bajar `b`**. Inclinar más la recta (subir `w`) sube todas las predicciones (las temperaturas son de 14 a 36 °C), así que hay que compensar bajándola entera con `b`.

📝 Ese cañón estrecho le complicará la vida al excursionista de la Lección 3.3. Allí verás cómo se arregla.
</details>

---
## 5. 📖 ¿Y en clasificación? La entropía cruzada

En clasificación (¿spam o no spam?) el modelo no da un número como "180 helados", sino una **probabilidad**: *"estoy 90 % seguro de que es spam"*. ¿Cómo medimos el error de una probabilidad?

La **entropía cruzada** (*cross-entropy*, *log loss*) mira **qué probabilidad le dio el modelo a la respuesta correcta** y castiga así:

| Probabilidad que dio a la respuesta correcta | Castigo |
|---|---|
| 99 % (seguro y correcto) | casi 0 |
| 50 % (no sabe) | moderado |
| 1 % (seguro… y equivocado) | **enorme** |

En fórmula, el castigo es $-\log(p)$, donde $p$ es la probabilidad que el modelo le dio a la respuesta correcta. No necesitas recordarla: mira la curva.

In [ ]:
p = np.linspace(0.005, 1, 300)
plt.plot(p, -np.log(p), lw=2)
for pi in [0.99, 0.5, 0.01]:
    plt.plot(pi, -np.log(pi), "ro"); plt.annotate(f"p={pi} → {-np.log(pi):.2f}", (pi, -np.log(pi)), xytext=(8, 8), textcoords="offset points")
plt.xlabel("Probabilidad que el modelo le dio a la respuesta CORRECTA"); plt.ylabel("Castigo (−log p)")
plt.title("Entropía cruzada: equivocarse con seguridad sale carísimo"); plt.grid(alpha=.3); plt.show()

### 🔮 Ejercicio 7 — ¿Quién paga más?
Un correo **sí es spam**. Tres modelos dijeron:
- Modelo A: "90 % spam"
- Modelo B: "60 % spam"
- Modelo C: "1 % spam" (o sea, 99 % seguro de que **no** es spam)

Ordénalos de menor a mayor castigo. ¿Cuántas veces más paga C que A, aproximadamente?

✍️ *Tu respuesta:*

In [ ]:
for nombre, p_spam in [("A", 0.90), ("B", 0.60), ("C", 0.01)]:
    print(f"Modelo {nombre}: dijo {p_spam:.0%} spam → castigo = {-np.log(p_spam):.3f}")

<details><summary><b>👀 Ver respuesta</b></summary>

**A (0.105) < B (0.511) < C (4.605)**. C paga unas **44 veces** más que A.

Por eso la entropía cruzada enseña al modelo a ser **humilde cuando no sabe**: decir "60 %" cuando dudas sale barato; decir "1 %" y fallar sale carísimo.
</details>

### 📖 ¿Por qué no usar directamente la exactitud como pérdida?
La exactitud (% de aciertos, Lección 4.2) es lo que solemos reportar. Pero para **entrenar** tiene un problema grave: casi siempre está **plana**. Si mueves un poquito una perilla, ningún ejemplo cambia de "acierto" a "fallo", así que la exactitud **no cambia** y el modelo no sabe hacia dónde moverse.

Comprobémoslo con un clasificador muy simple que decide "spam" según cuántas veces aparece la palabra *gratis*, y que tiene una sola perilla `w`.

In [ ]:
rng_spam = np.random.default_rng(3)
n_gratis = np.concatenate([rng_spam.poisson(1, 40), rng_spam.poisson(4, 40)])   # veces que aparece "gratis"
es_spam = np.array([0] * 40 + [1] * 40)

def prob_spam(x, w):
    return 1 / (1 + np.exp(-(w * x - 5)))      # convierte el puntaje en una probabilidad entre 0 y 1

ws = np.linspace(0, 4, 400)
exactitud = [np.mean((prob_spam(n_gratis, w) >= 0.5) == es_spam) for w in ws]
entropia = [np.mean(-np.log(np.where(es_spam == 1, prob_spam(n_gratis, w), 1 - prob_spam(n_gratis, w)))) for w in ws]

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
a1.plot(ws, exactitud, lw=2); a1.set_title("Exactitud vs. perilla w"); a1.set_xlabel("w"); a1.grid(alpha=.3)
a2.plot(ws, entropia, lw=2, color="tab:red"); a2.set_title("Entropía cruzada vs. perilla w"); a2.set_xlabel("w"); a2.grid(alpha=.3)
plt.show()

### ✋ Ejercicio 8
Imagina que eres el excursionista de la Lección 3.3 y solo puedes sentir la pendiente bajo tus pies.
1. ¿En cuál de los dos paisajes podrías saber hacia dónde caminar desde **cualquier** punto?
2. ¿Qué le pasa al excursionista en la gráfica de exactitud si está parado en `w = 0.3`? ¿Y en `w = 3`?

✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. En el de **entropía cruzada**: es una curva suave, con pendiente en casi todas partes que apunta hacia el mínimo.
2. La exactitud es una **escalera**: en `w = 0.3` el terreno es completamente plano (el modelo no marca ningún correo como spam y la exactitud se queda en 50 % al mover un poco `w`). En `w = 3` también es plano, aunque *no* está en el mejor punto: la entropía cruzada, en cambio, sí le dice que baje `w` hacia ≈ 2. El excursionista en la escalera no siente ninguna pendiente y **no sabe hacia dónde ir**.

Conclusión: **entrenamos con una pérdida suave** (entropía cruzada) y **reportamos la métrica que entiende el negocio** (exactitud, recall… Módulo 4).
</details>

---
## 6. 📖 Elegir la pérdida también es una decisión tuya

La función de pérdida es un **hiperparámetro**: tú decides qué errores importan más. Las más comunes:

| Pérdida | Tipo de problema | Qué castiga | Úsala cuando… | En scikit-learn |
|---|---|---|---|---|
| **MSE** | Regresión | Errores grandes, mucho más que los pequeños | Los errores grandes son graves y los datos están limpios | `loss="squared_error"` (por defecto) |
| **MAE** | Regresión | Todos los errores por igual, según su tamaño | Hay atípicos que no quieres que dominen | `loss="absolute_error"` |
| **Huber** | Regresión | Como MSE en errores chicos, como MAE en grandes | Quieres un punto medio | `loss="huber"` |
| **Entropía cruzada** (log loss) | Clasificación | Probabilidades muy seguras y equivocadas | Casi siempre en clasificación | `loss="log_loss"` |

### ✋ Ejercicio 9 — Elige la pérdida
1. Una app predice el **tiempo de entrega** de comida. Llegar 5 minutos tarde molesta un poco; llegar 60 minutos tarde hace que el cliente borre la app.
2. Un sensor de temperatura de vez en cuando manda lecturas absurdas (−999 °C) por fallas eléctricas, y quieres predecir la temperatura de mañana.
3. Un modelo debe decir si una radiografía muestra una fractura, con una probabilidad.

✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. **MSE**: los retrasos grandes son mucho más graves que los pequeños, y el MSE los castiga en proporción al cuadrado.
2. **MAE** (o Huber): no quieres que las lecturas basura dominen el entrenamiento. (Y, mejor aún, limpia esos datos: Lección 2.2.)
3. **Entropía cruzada**: es clasificación con probabilidades.
</details>

---
## 7. 📖 Errores comunes y señales de alerta

| 🚨 Error | Por qué es un problema | Qué hacer |
|---|---|---|
| Usar MSE con atípicos sin revisar los datos | Unos pocos puntos raros "jalan" todo el modelo | Revisa los atípicos (Lección 2.2); prueba MAE o Huber |
| Comparar pérdidas de problemas distintos | Un MSE de 200 "helados²" y uno de 0.3 "°C²" no se pueden comparar | Compara solo modelos sobre **los mismos datos**, o usa métricas relativas (R², Lección 4.1) |
| Usar la exactitud para entrenar | Es plana: no da dirección | Entrena con entropía cruzada; reporta exactitud |
| Celebrar una pérdida de 0 en entrenamiento | Casi siempre significa que el modelo **memorizó** | Mira la pérdida en datos de **validación** (Lección 3.7) |
| Confundir la pérdida con la meta del negocio | Bajar el MSE no garantiza ganar más dinero | Elige la métrica según el costo real del error (Lección 4.8) |

### 🔮 Ejercicio 10
Un modelo de precios de casas reporta **MSE = 40,000,000,000**. Tu compañera dice que es un desastre. ¿Tiene razón? ¿Qué número calcularías para saberlo?

✍️ *Tu respuesta:*

In [ ]:
mse_casas = 40_000_000_000
print(f"RMSE = ${np.sqrt(mse_casas):,.0f}")

<details><summary><b>👀 Ver respuesta</b></summary>

**No se puede saber sin contexto.** El MSE está en *pesos al cuadrado*, por eso sale gigantesco. Su raíz (RMSE) es **$200,000**: un error típico de doscientos mil pesos. Si las casas cuestan $5,000,000, es un 4 % (bastante bien); si cuestan $400,000, es un desastre. Siempre traduce la pérdida a **unidades del negocio**.
</details>

---
## 8. 🧠 Resumen en 5 puntos
1. La **función de pérdida** resume todos los errores del modelo en **un solo número**: el que el entrenamiento intenta bajar.
2. **MAE** = promedio del tamaño del error (trata a todos por igual). **MSE** = promedio del error al cuadrado (castiga mucho los errores grandes y es sensible a atípicos).
3. En clasificación se usa la **entropía cruzada**: castiga muchísimo equivocarse con seguridad.
4. Para cada posición de las perillas hay una pérdida: juntas forman un **paisaje** con valles. Entrenar es **bajar al fondo del valle**.
5. Elegir la pérdida es un **hiperparámetro**: decide qué errores te importan. Se entrena con una pérdida suave y se reporta la métrica del negocio.

---
## 9. 🏋️ Ejercicios finales

### 9.1 Repaso rápido (opción múltiple)

**1.** Los errores de un modelo son: +3, −3, +3, −3. ¿Cuánto valen el MAE y el MSE?
- a) MAE 0, MSE 0 · b) MAE 3, MSE 9 · c) MAE 3, MSE 3 · d) MAE 12, MSE 36

**2.** Agregas a tus datos un único punto con un error enorme. ¿Qué pérdida sube **más** (en proporción)?
- a) MAE · b) MSE · c) Las dos igual · d) Ninguna

**3.** Un modelo dice "99 % seguro: este paciente NO tiene la enfermedad", y sí la tiene. Con entropía cruzada, el castigo es…
- a) casi 0 · b) moderado · c) muy grande · d) negativo

**4.** ¿Por qué no se entrena directamente con la exactitud?
- a) Porque es muy lenta de calcular · b) Porque es plana: pequeños cambios en las perillas no la mueven · c) Porque solo sirve para regresión · d) Porque siempre vale 50 %

**5.** El paisaje de la pérdida es…
- a) Un dibujo de los datos · b) La pérdida para cada posible posición de las perillas · c) La gráfica de la recta · d) La lista de hiperparámetros

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **b)** El tamaño de cada error es 3 (MAE = 3) y cada cuadrado es 9 (MSE = 9).
2. **b)** Al elevar al cuadrado, un error enorme se vuelve gigantesco.
3. **c)** Le dio 1 % a la respuesta correcta: −log(0.01) ≈ 4.6, el castigo más fuerte de la tabla.
4. **b)** El excursionista no siente pendiente en un terreno plano.
5. **b)**
</details>

### 9.2 Preguntas para pensar (sin ejecutar)

**A.** Un hospital predice cuántas camas necesitará mañana. Quedarse **corto** es grave (pacientes sin cama); sobrar camas solo cuesta algo de dinero. ¿MAE o MSE te parecen suficientes? ¿Qué le cambiarías a la pérdida?

**B.** ¿Por qué crees que en la pérdida promediamos los errores (dividimos entre el número de ejemplos) en lugar de sumarlos sin más?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuestas</b></summary>

**A.** Ninguna de las dos distingue **faltar** de **sobrar**: ambas tratan igual un error de +5 que uno de −5. Lo ideal sería una pérdida **asimétrica**, que castigue más quedarse corto (por ejemplo, multiplicar por 5 los errores por debajo). Existe y se llama *pérdida cuantil* (*quantile loss*). Es la misma idea del proyecto del Módulo 5: el costo del error manda.

**B.** Para que la pérdida no dependa de **cuántos** datos tienes. Con la suma, 1,000 días darían una pérdida 10 veces mayor que 100 días aunque el modelo fuera igual de bueno. El promedio permite comparar y, además, mantiene estable el tamaño de los pasos del entrenamiento (Lección 3.4).
</details>

### 9.3 🧩 Práctica integradora: ¿qué app del clima es mejor? 🌧️

Dos apps del clima dieron, durante 12 días, la **probabilidad de lluvia**. Tú anotaste si llovió (1) o no (0). Quieres decidir cuál app conservar.

**Paso 1 ▶️ — Los datos**

In [ ]:
llovio  = np.array([1,   0,   0,   1,   1,   0,   0,   1,   0,   0,   1,   0])
app_sol = np.array([.60, .40, .45, .55, .65, .35, .40, .60, .45, .30, .55, .40])   # "App Sol": siempre tibia
app_nube = np.array([.95, .05, .10, .90, .97, .03, .02, .85, .08, .01, .90, .99])  # "App Nube": muy segura

print("Días de lluvia:", llovio.sum(), "de", len(llovio))

**Paso 2 ▶️ — Exactitud.** Si cada app "dice que lloverá" cuando su probabilidad es ≥ 50 %, ¿cuántos días acierta cada una?

In [ ]:
for nombre, prob in [("App Sol", app_sol), ("App Nube", app_nube)]:
    aciertos = np.mean((prob >= 0.5) == llovio)
    print(f"{nombre}: exactitud = {aciertos:.0%}")

<details><summary><b>👀 Ver respuesta</b></summary>

**App Sol: 100 %** (acierta los 12 días, aunque siempre con probabilidades tibias). **App Nube: 92 %** (11 de 12): falló el último día, en el que dijo 99 % de lluvia y no llovió. Según la exactitud, App Sol gana.
</details>

**Paso 3 🔮 — Antes de calcular.** Con lo que sabes de la entropía cruzada, ¿qué app crees que tendrá **menor** pérdida? Piensa en dos cosas: App Nube es muy segura cuando acierta… pero tiene un día catastrófico.

✍️ *Mi apuesta:*

**Paso 4 ▶️ — Calcula la entropía cruzada**

In [ ]:
def entropia_cruzada(real, prob):
    p_correcta = np.where(real == 1, prob, 1 - prob)     # probabilidad que le dio a lo que SÍ pasó
    return np.mean(-np.log(p_correcta))

for nombre, prob in [("App Sol", app_sol), ("App Nube", app_nube)]:
    print(f"{nombre}: entropía cruzada = {entropia_cruzada(llovio, prob):.3f}")

<details><summary><b>👀 Ver respuesta</b></summary>

**App Nube: ≈ 0.45** · **App Sol: ≈ 0.51**. ¡App Nube gana pese a tener menor exactitud! Sus 11 aciertos muy seguros (castigo casi 0) compensan su día catastrófico. App Sol nunca falla, pero siempre duda: paga un castigo moderado *todos* los días.
</details>

**Paso 5 ✏️ — El peso del día catastrófico.** Cambia el último valor de App Nube de `.99` a `.60` (sigue equivocándose, pero con menos seguridad) y vuelve a ejecutar la celda del Paso 4. ¿Cuánto baja su pérdida?

In [ ]:
app_nube_humilde = app_nube.copy()
app_nube_humilde[-1] = 0.60       # ✏️ prueba también 0.999 o 0.51
print(f"App Nube humilde: entropía cruzada = {entropia_cruzada(llovio, app_nube_humilde):.3f}")

<details><summary><b>👀 Ver respuesta</b></summary>

Baja de **≈ 0.45 a ≈ 0.14**. Un solo día con un error "seguro" (99 %) aportaba **casi el 90 %** de toda su pérdida. Si lo cambias a 0.999, se dispara a ≈ 0.64 y App Sol vuelve a ganar.
</details>

**Paso 6 ✋ — Decide como negocio.** Organizas bodas al aire libre: si la app dice que habrá lluvia, rentas una carpa ($$$). ¿Con qué app te quedas? ¿Qué te dice cada métrica?

✍️ *Mi decisión:*

<details><summary><b>👀 Ver respuesta</b></summary>

Una respuesta razonable: **App Nube**. Sus probabilidades son **útiles para decidir**: un 95 % o un 3 % te dicen con claridad si rentar la carpa. App Sol acierta siempre la dirección, pero con un 55 % o un 45 % nunca sabes si vale la pena el gasto. La **exactitud** solo mira si cruzas el 50 %; la **entropía cruzada** mide si puedes **confiar** en las probabilidades.

Pero ojo con su día catastrófico: con 12 días tienes muy poca evidencia. Pide más historial antes de decidir (Lección 4.6).
</details>

---
## 📚 Para profundizar
- 📺 **Video:** StatQuest, *"Cross Entropy"* y *"Log Loss"* (inglés, con subtítulos).
- 📘 **Lectura:** Google, *Machine Learning Crash Course*, sección *Loss*.
- 📘 **Documentación:** scikit-learn, *Metrics and scoring* (`mean_absolute_error`, `mean_squared_error`, `log_loss`).

---
⬅️ **Anterior:** [3.1 El modelo como "función con perillas"](3.1-funcion-con-perillas.ipynb) · ➡️ **Siguiente:** [3.3 Optimización: el excursionista en la niebla (descenso de gradiente)](3.3-descenso-de-gradiente.ipynb)